# Change-point detection lab
Compare a sequential CUSUM alarm, greedy binary segmentation and penalized PELT on a synthetic series with two mean shifts. These are teaching baselines, not calibrated event detectors.

In [ ]:
import sys
sys.path.insert(0, "implementations/python/change-point-detection")
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / 'implementations/python/change-point-detection'))
from detectors import cusum, binary_segmentation, pelt
import random
random.seed(7)
series = [random.gauss(0, 0.35) for _ in range(40)] + [random.gauss(2, 0.35) for _ in range(35)] + [random.gauss(-1, 0.35) for _ in range(35)]
len(series), series[:3]

## Compare segmentation outputs
The true boundaries are 40 and 75. Tune the penalty and minimum segment size; do not treat a visually plausible fit as proof of statistical significance.

In [ ]:
print('Binary segmentation:', binary_segmentation(series, max_changes=2, min_size=8))
print('PELT:', pelt(series, penalty=8.0, min_size=8))
print('CUSUM alarms:', cusum(series, target=0.0, threshold=4.0))

## Interpretation checklist
- Do boundaries persist under modest penalty/minimum-length changes?
- Could seasonality, missing acquisitions or sensor harmonization create the apparent shift?
- Are neighbouring pixels spatially dependent?
- What independent evidence supports a real process transition?

## Bayesian online change-point detection
This baseline assumes a Gaussian likelihood with known observation variance, a Normal prior on each segment mean and a constant hazard. It is not the unknown-variance Student-t model.

In [ ]:
from online import bocpd_normal_mean
stream = [0.0] * 20 + [4.0] * 8
cp_probability, map_run_length = bocpd_normal_mean(
    stream, observation_variance=0.05, hazard=0.05
)
list(zip(cp_probability, map_run_length))[-10:]